# 22b · Randstorm contrast under strict migration outcomes

Inputs: `key_pubform` (21); `q22_out`, `q22_fate`, `q22_spend` and the other inputs of 22

Outputs: `out22b/`, zipped as `out22b.zip`

Re-estimates the Randstorm contrast of 21 (group-L keys first seen 2012-04 to 2014-03, uncompressed U vs compressed C public keys) and the pooled quantum-event contrast (L − C) on the strict outcomes of 22 and two further outcomes. The `randstorm` row in the output of 22 is an L − C contrast and is not used.

Outcomes (key-weighted cumulative share over days 0…+30, capped at 1):

- `v_all`, `v_sweep`, `v_sweep_kept`, `v_unspent`: as in 22
- `v_noexp`: not a CoinJoin, and no output of the transaction goes to an exposed address or an exchange (several new destination addresses allowed)
- `v_noexp_kept`: `v_noexp`, and afterwards untouched or kept in use

Randomization-inference p-values are reported in two forms: `p_ri` (|placebo| ≥ |β|, as in 20–22) and `p_eq` (equal-tailed, consistent with the 90% interval). Pooled effects also report `null_mean`, the mean of the placebo distribution.

In [ ]:
# Configuration
PROJECT_ID = "your-gcp-project"
DATASET    = "qbtc_raw"
LOCATION   = "US"

AUTO_RUN_USD      = 5.0
MAX_USD_PER_QUERY = 10.0
MAX_USD_TOTAL     = 100.0
PRICE_PER_TIB_USD = 6.25
CONFIRM           = False
FORCE_REBUILD     = False

FREEZE_DATE   = "2026-09-22"   # freeze date
POST_MAX      = 30             # outcomes: relative days 0...+30 (as in 05)
DORMANT_DAYS  = 30             # y1 strict: unspent for the next 30 days (as in 04)
CONSOL_MIN_IN = 20             # spending transaction with at least this many inputs -> consolidation
CJ_MIN_KEYS   = 3              # CoinJoin: at least 3 input keys
CJ_MIN_SAME   = 3              #           and at least 3 outputs of equal value

H_MAIN      = 90               # as in 20: local linear bandwidth (days)
BANDWIDTHS  = [60, 90, 120]
POOLS       = ["main", "sym30"]
DELTA2      = 0.0235 / 4       # equivalence margin: 1/4 of the Randstorm effect (21)
REF_POOLED  = 8.184148315025512e-05   # main result of 20: LC, key-weighted, y1 strict, +30, main pool, h = 90
# Randstorm contrast of 21 (uncompressed U vs compressed C)
RS_WIN = ("2012-04-01", "2014-03-31")  # treated cohort: first-seen dates (main window of 21)
REF_RS2_THETA    = 0.0236255607294543     # 21: randstorm, RS2, y1 strict, +30, key-weighted, theta
REF_RS2_BETA     = 0.0235279128448018     # 21: same, beta (main pool, h = 90)
REF_RS2ALL_THETA = 0.0042476989230802     # 21: theta of RS2_all
REF_UC_POOLED    = -0.0005383462026213    # 21: pooled U-C effect on the 6 main events
N_PAY, N_SWEEP, N_CJ, SEED = 30, 10, 3, 20260925   # sample: 30 payment-or-change, 10 sweeps, <=3 CoinJoins
OUT_DIR = "out22b"

try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

from google.cloud import bigquery
import json, os, io, re, time, zipfile, datetime as dt
import numpy as np
import pandas as pd

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
DS  = f"{PROJECT_ID}.{DATASET}"
PUB = "bigquery-public-data.crypto_bitcoin"
os.makedirs(OUT_DIR, exist_ok=True)

LEDGER_PATH, JOBS_PATH = "build_ledger.json", "jobs.json"
LEDGER = json.load(open(LEDGER_PATH)) if os.path.exists(LEDGER_PATH) else {"jobs": [], "usd_total": 0.0}
JOBS   = json.load(open(JOBS_PATH))   if os.path.exists(JOBS_PATH)   else []

for t in ["flow_days", "flow_key_tx", "flow_tx_inputs", "flow_tx_outputs", "dest_key_info", "exchange_keys",
          "panel_keys", "placebo_draws_strict", "key_pubform", "q22_out", "q22_fate", "q22_spend"]:
    client.get_table(f"{DS}.{t}")        # raises if a table is missing
print("Tables from 21 and 22 found. Spent in this Colab session: $%.2f / $%.0f" % (LEDGER["usd_total"], MAX_USD_TOTAL))

In [ ]:
# Helpers
def usd(gb): return gb / 1024 * PRICE_PER_TIB_USD

def dry_run(sql):
    job = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = job.total_bytes_processed / 2**30
    return gb, usd(gb)

def table_rows(name):
    try:
        return client.get_table(f"{DS}.{name}").num_rows
    except Exception:
        return None

def table_info(name):
    t = client.get_table(f"{DS}.{name}")
    return {"table": name, "rows": t.num_rows, "GB": round(t.num_bytes / 2**30, 2)}

def build(name, sql, force=False):
    assert not re.search(r"CLUSTER BY[^\n]*\b(key_id|txid|spending_txid)\b", sql), f"[{name}] cannot cluster on a BYTES column"
    n = table_rows(name)
    if n and not (FORCE_REBUILD or force):
        print(f"[{name}] exists, {n:,} rows; skipped (rebuild with build(..., force=True))")
        return True
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{name}] estimated ${cost:.2f}, above the per-query limit ${MAX_USD_PER_QUERY}")
    if LEDGER["usd_total"] + cost > MAX_USD_TOTAL:
        raise RuntimeError(f"[{name}] total would exceed ${MAX_USD_TOTAL}")
    if cost > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}, above the auto-run limit ${AUTO_RUN_USD}: set CONFIRM=True and re-run this cell (tables already built are skipped)")
    print(f"[{name}] estimated {gb:,.0f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    JOBS.append({"table": name, "job_id": job.job_id, "location": LOCATION,
                 "submitted_at": dt.datetime.now(dt.timezone.utc).isoformat()})
    json.dump(JOBS, open(JOBS_PATH, "w"), indent=2)
    print(f"[{name}] submitted job {job.job_id} (runs on the server; finishes even if Colab disconnects)")
    t0 = time.time()
    job.result()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": name, "job_id": job.job_id, "billed_gb": round(billed, 1),
                           "usd": round(usd(billed), 2), "seconds": round(time.time() - t0, 1)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    print(f"[{name}] done, {billed:,.0f} GB, {LEDGER['jobs'][-1]['seconds']}s", table_info(name))
    return True

def need(*names):
    missing = [n for n in names if not table_rows(n)]
    assert not missing, f"These tables are not built yet: {missing} (run the earlier cells first)"

def q(sql):
    return client.query(sql).to_dataframe()

def q_guarded(sql, label):
    """Queries that do not build a table are also priced first and stop above the auto-run limit."""
    gb, cost = dry_run(sql)
    if cost > MAX_USD_PER_QUERY:
        raise RuntimeError(f"[{label}] estimated ${cost:.2f}, above the per-query limit")
    if cost > AUTO_RUN_USD and not CONFIRM:
        raise RuntimeError(f"[{label}] estimated ${cost:.2f}, above the auto-run limit: set CONFIRM=True")
    print(f"[{label}] estimated {gb:,.1f} GB ~ ${cost:.2f}")
    job = client.query(sql, job_config=bigquery.QueryJobConfig(
        maximum_bytes_billed=int(gb * 1.25 * 2**30) + 10 * 2**30, use_query_cache=False))
    df = job.result().to_dataframe()
    billed = (job.total_bytes_billed or 0) / 2**30
    LEDGER["jobs"].append({"table": label, "job_id": job.job_id, "billed_gb": round(billed, 1), "usd": round(usd(billed), 2)})
    LEDGER["usd_total"] = round(LEDGER["usd_total"] + usd(billed), 2)
    json.dump(LEDGER, open(LEDGER_PATH, "w"), indent=2)
    return df

def save(df, name, **kw):
    df.to_csv(os.path.join(OUT_DIR, name), **kw)

In [ ]:
# Dates and constants
FREEZE = pd.Timestamp(FREEZE_DATE)
F  = FREEZE.strftime("%Y-%m-%d")
F1 = (FREEZE + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
fr = q(f"SELECT MIN(date) AS lo, MAX(date) AS hi FROM `{DS}.flow_days`")
LO, HI = pd.Timestamp(fr.lo.iloc[0]).strftime("%Y-%m-%d"), pd.Timestamp(fr.hi.iloc[0]).strftime("%Y-%m-%d")
assert HI <= F, f"last day of flow_days {HI} is after the freeze date {F}"
print(f"Dates tagged in 04: {LO} ... {HI}; freeze date {F}")

## Step 1 · Output roles in candidate transactions, `q22_outrole`

Candidate transactions and classification rules as in step 1 of 22, keeping all outputs. Roles: change / exchange / exposed / y1_strict / unexposed_other / no_key.

Check: y1-strict output count and value per transaction match `q22_out`.

In [ ]:
SQL_OUTROLE = f"""
CREATE OR REPLACE TABLE `{DS}.q22_outrole`
PARTITION BY created_date
AS
WITH cand AS (   -- as in step 1 of 22: at least one risk-set key has a y1-strict outflow in the transaction
  SELECT DISTINCT txid, date
  FROM `{DS}.flow_key_tx`
  WHERE y1_strict > 0 AND date BETWEEN DATE '{LO}' AND DATE '{HI}'
),
tx_in AS (
  SELECT i.spending_txid AS txid,
         ARRAY_AGG(DISTINCT i.key_id IGNORE NULLS) AS in_keys,
         COUNT(*) AS n_in,
         COUNT(DISTINCT i.key_id) AS n_in_keys
  FROM `{DS}.flow_tx_inputs` i
  JOIN cand c ON c.txid = i.spending_txid AND c.date = i.spent_date
  WHERE i.spent_date BETWEEN DATE '{LO}' AND DATE '{HI}'
  GROUP BY i.spending_txid
),
o AS (   -- classification as in 04 and 22
  SELECT o.txid, o.vout, o.created_date, o.sats, o.type, o.address, o.out_spent_date,
    t.n_in, t.n_in_keys,
    IFNULL(o.dest_key_id IN UNNEST(t.in_keys), FALSE)                               AS is_change,
    ex.key_id IS NOT NULL                                                            AS to_ex,
    (o.type IN ('pubkey', 'multisig', 'witness_v1_taproot')
       OR IFNULL(dk.first_exposed_date <= o.created_date, FALSE))                    AS exposed,
    o.dest_key_id IS NOT NULL                                                        AS keyed,
    IFNULL(o.addr_first_seen_height = o.created_height, FALSE)                       AS is_new_addr,
    IFNULL(o.addr_first_seen_height = o.created_height, FALSE)
      AND o.type IN ('pubkeyhash', 'witness_v0_keyhash', 'scripthash', 'witness_v0_scripthash')
      AND (o.out_spent_date IS NULL
           OR o.out_spent_date > DATE_ADD(o.created_date, INTERVAL {DORMANT_DAYS} DAY))  AS strict_ok
  FROM `{DS}.flow_tx_outputs` o
  JOIN cand c ON c.txid = o.txid AND c.date = o.created_date
  JOIN tx_in t ON t.txid = o.txid
  LEFT JOIN `{DS}.dest_key_info` dk ON dk.key_id = o.dest_key_id
  LEFT JOIN `{DS}.exchange_keys` ex ON ex.key_id = o.dest_key_id
  WHERE o.created_date BETWEEN DATE '{LO}' AND DATE '{HI}'
)
SELECT txid, vout, created_date, sats, type, address, out_spent_date, n_in, n_in_keys, is_new_addr,
  CASE WHEN is_change THEN 'change'
       WHEN to_ex THEN 'y2'
       WHEN exposed THEN 'y0'
       WHEN keyed AND strict_ok THEN 'y1s'
       WHEN keyed THEN 'y1o'
       ELSE 'oth' END AS role
FROM o
"""
build("q22_outrole", SQL_OUTROLE)

# check against q22_out
chk1 = q(f"""
WITH a AS (SELECT txid, COUNTIF(role = 'y1s') AS n_y1s, SUM(IF(role = 'y1s', sats, 0)) AS s_y1s
           FROM `{DS}.q22_outrole` GROUP BY txid),
     b AS (SELECT txid, COUNT(*) AS n_y1s, SUM(sats) AS s_y1s FROM `{DS}.q22_out` GROUP BY txid)
SELECT COUNT(*) AS n_tx,
       COUNTIF(a.txid IS NULL OR b.txid IS NULL) AS unmatched,
       COUNTIF(IFNULL(a.n_y1s, 0) != IFNULL(b.n_y1s, 0) OR IFNULL(a.s_y1s, 0) != IFNULL(b.s_y1s, 0)) AS mismatched
FROM a FULL OUTER JOIN b USING (txid)
""")
display(chk1)
assert chk1.unmatched.iloc[0] == 0 and chk1.mismatched.iloc[0] == 0, "y1-strict outputs in q22_outrole do not match q22_out"
print("Step 1 check passed: y1-strict outputs of each transaction match 22")
display(q(f"""SELECT role, COUNT(*) AS n_out, ROUND(SUM(sats) / 1e8, 1) AS btc
              FROM `{DS}.q22_outrole` GROUP BY role ORDER BY btc DESC"""))

## Step 2 · Outcome cells

- L−C cells: as in 22 (event × group × CEM cell).
- U−C cells: the keys of 21 (group L, hash160, first seen 2012-04 to 2014-03, known public-key format) by the four strata of 21 (balance, coin age, last incoming transfer, UTXO count), plus `rec_old` (no incoming transfer in the previous 365 days; excluded from RS2).

In [ ]:
need("q22_outrole", "q22_out", "q22_fate", "q22_spend")
TXROLE_SQL = f"""
  SELECT txid, COUNT(*) AS n_out,
    COUNTIF(role = 'change') AS n_change, COUNTIF(role = 'y2') AS n_y2, COUNTIF(role = 'y0') AS n_y0,
    COUNTIF(role = 'y1s') AS n_y1s, COUNTIF(role = 'y1o') AS n_y1o, COUNTIF(role = 'oth') AS n_oth,
    SUM(IF(role = 'change', sats, 0)) AS s_change, SUM(IF(role = 'y2', sats, 0)) AS s_y2,
    SUM(IF(role = 'y0', sats, 0)) AS s_y0, SUM(IF(role = 'y1s', sats, 0)) AS s_y1s,
    SUM(IF(role = 'y1o', sats, 0)) AS s_y1o, SUM(IF(role = 'oth', sats, 0)) AS s_oth
  FROM `{DS}.q22_outrole` GROUP BY txid"""
FATE_SQL = f"""
  SELECT o.txid, o.vout, o.sats, o.pattern,
    CASE WHEN o.out_spent_date IS NULL THEN 'unspent'
         WHEN s.to_ex THEN 'to_exchange'
         WHEN s.n_in >= {CONSOL_MIN_IN} THEN 'consolidated'
         WHEN s.h IS NULL THEN 'spent_unknown'
         ELSE 'spent_kept' END AS fate,
    tr.n_y0, tr.n_y2
  FROM `{DS}.q22_out` o
  LEFT JOIN `{DS}.q22_fate` f USING (txid, vout)
  LEFT JOIN `{DS}.q22_spend` s ON s.h = LOWER(TO_HEX(f.spending_txid))
  LEFT JOIN ({TXROLE_SQL}) tr ON tr.txid = o.txid"""
OUTC = {   # outcome -> y1-strict outputs counted in each transaction (the first four as in 22)
    "v_all":        "TRUE",
    "v_sweep":      "pattern = 'sweep'",
    "v_sweep_kept": "pattern = 'sweep' AND fate IN ('unspent', 'spent_kept')",
    "v_unspent":    "pattern != 'coinjoin' AND fate = 'unspent'",
    "v_noexp":      "pattern != 'coinjoin' AND n_y0 = 0 AND n_y2 = 0",
    "v_noexp_kept": "pattern != 'coinjoin' AND n_y0 = 0 AND n_y2 = 0 AND fate IN ('unspent', 'spent_kept')",
}
cls_cols = ",\n    ".join(f"SUM(IF({cond}, sats, 0)) AS s_{k}" for k, cond in OUTC.items())
kf_cols  = ",\n    ".join(f"SUM(f.share * c.s_{k}) AS a_{k}" for k in OUTC)
k_cols   = ",\n    ".join(f"LEAST(1.0, IFNULL(kf.a_{k}, 0) / pk.sats) AS {k}" for k in OUTC)
agg_cols = ",\n  ".join(f"SUM({k}) AS {k}" for k in OUTC)

# U / C keys of 21: group L, hash160, first seen within RS_WIN, known public-key format
PK_UC = f"""
  SELECT pk.event, pk.event_set, pk.event_date, pk.key_id, pk.sats, pf.pubform AS arm,
    CONCAT(pk.bal_bin, '|', pk.age_bin, '|', pk.rec_bin, '|', pk.nutxo_bin) AS stratum,
    pk.rec_bin = '365d+' AS rec_old
  FROM `{DS}.panel_keys` pk
  JOIN `{DS}.key_pubform` pf ON pf.key_id = pk.key_id
  WHERE pk.grp = 'L' AND pk.family = 'hash160' AND pf.pubform IN ('U', 'C')
    AND pk.first_seen_date BETWEEN DATE '{RS_WIN[0]}' AND DATE '{RS_WIN[1]}'"""

def cells_sql(pk_sql, group_cols, key_filter=""):
    return f"""
WITH fd AS (
  SELECT event, rel_day, date FROM `{DS}.flow_days` WHERE rel_day BETWEEN 0 AND {POST_MAX}
),
fo AS ({FATE_SQL}
),
c AS (
  SELECT txid,
    {cls_cols}
  FROM fo GROUP BY txid
),
pk AS ({pk_sql}
),
kf AS (
  SELECT fd.event, f.key_id,
    {kf_cols}
  FROM `{DS}.flow_key_tx` f
  JOIN fd ON fd.date = f.date
  JOIN c ON c.txid = f.txid
  WHERE f.y1_strict > 0 AND f.date BETWEEN DATE '{LO}' AND DATE '{HI}' {key_filter}
  GROUP BY fd.event, f.key_id
),
k AS (
  SELECT pk.*,
    {k_cols}
  FROM pk
  LEFT JOIN kf ON kf.event = pk.event AND kf.key_id = pk.key_id
)
SELECT event, ANY_VALUE(event_set) AS event_set, ANY_VALUE(event_date) AS event_date, {group_cols},
  COUNT(*) AS n_keys, SUM(sats) / 1e8 AS bal_btc,
  {agg_cols}
FROM k
GROUP BY event, {group_cols}
"""

PK_LC = f"""
  SELECT event, event_set, event_date, key_id, sats, grp, cem_id FROM `{DS}.panel_keys`"""
cells = q_guarded(cells_sql(PK_LC, "grp, cem_id"), "q22b_cells_lc")
cells_uc = q_guarded(cells_sql(PK_UC, "arm, stratum, rec_old",
                               f"AND f.key_id IN (SELECT key_id FROM `{DS}.key_pubform` WHERE pubform IN ('U', 'C'))"),
                     "q22b_cells_uc")
for d_ in (cells, cells_uc):
    d_["event_date"] = pd.to_datetime(d_.event_date)
for d_, nm in [(cells, "L−C"), (cells_uc, "U−C")]:
    for k in OUTC:
        if k != "v_all":
            assert (d_[k] <= d_.v_all + 1e-9).all(), f"{nm}: {k} must not exceed v_all"
    assert (d_.v_noexp_kept <= d_.v_noexp + 1e-9).all() and (d_.v_sweep_kept <= d_.v_sweep + 1e-9).all()
print(f"L-C cells {len(cells):,} rows; U-C cells {len(cells_uc):,} rows")
sz = cells_uc[cells_uc.event == "randstorm"].groupby(["rec_old", "arm"]).n_keys.sum().unstack("arm")
print("Randstorm U / C keys (rec_old = True: no incoming transfer in the past 365 days; not used in RS2):"); display(sz)
cells.to_csv(os.path.join(OUT_DIR, "q22b_cells_lc.csv.gz"), index=False, compression="gzip")
cells_uc.to_csv(os.path.join(OUT_DIR, "q22b_cells_uc.csv.gz"), index=False, compression="gzip")

## Step 3 · Estimation

Estimator and inference of 20 and 21. Contrasts: `LC` (quantum events), `UC_RS2` (Randstorm, main), `UC_RS2_all` (no recent-inflow restriction).

Regression check: `v_all` reproduces the pooled L−C effect of 20 and the Randstorm θ, β and pooled U−C effect of 21 (tolerance 1e-8).

In [ ]:
def theta_lc(col):   # as in 22
    c = cells[cells.grp.isin(["L", "C"])]
    p = c.pivot_table(index=["event", "cem_id"], columns="grp", values=["n_keys", col], aggfunc="sum", fill_value=0)
    nL, nC = p[("n_keys", "L")], p[("n_keys", "C")]
    sL, sC = p[(col, "L")], p[(col, "C")]
    m = (nL > 0) & (nC > 0)
    g = pd.DataFrame({"den": nL.where(m, 0), "T": sL.where(m, 0),
                      "C": (nL / nC.where(nC > 0)).fillna(0).where(m, 0) * sC}).groupby(level="event").sum()
    return (g["T"] - g["C"]) / g["den"], g["T"] / g["den"]

def theta_uc(col, rs2=True):   # as theta in 21: a stratum counts only if it has both U and C keys
    c = cells_uc[~cells_uc.rec_old] if rs2 else cells_uc
    p = c.pivot_table(index=["event", "stratum"], columns="arm", values=["n_keys", col], aggfunc="sum", fill_value=0)
    nU, nC = p[("n_keys", "U")], p[("n_keys", "C")]
    sU, sC = p[(col, "U")], p[(col, "C")]
    m = (nU > 0) & (nC > 0)
    g = pd.DataFrame({"den": nU.where(m, 0), "T": sU.where(m, 0),
                      "C": (nU / nC.where(nC > 0)).fillna(0).where(m, 0) * sC}).groupby(level="event").sum()
    g = g[g.den > 0]
    return (g["T"] - g["C"]) / g["den"], g["T"] / g["den"]

ev = (cells.groupby("event").agg(event_set=("event_set", "first"), event_date=("event_date", "first"))
           .reset_index().sort_values("event_date").reset_index(drop=True))
T0 = pd.Timestamp("2020-01-01")
tday = ((ev.event_date - T0).dt.days).to_numpy(float)
EV_IDX = {e: i for i, e in enumerate(ev.event)}
EVL = ev.event.tolist()

CONTRASTS = {"LC": lambda k: theta_lc(k), "UC_RS2": lambda k: theta_uc(k, True), "UC_RS2_all": lambda k: theta_uc(k, False)}
diff_rows, lev_rows = {}, {}
for cn, fn in CONTRASTS.items():
    for k in OUTC:
        d_, l_ = fn(k)
        diff_rows[(cn, k)], lev_rows[(cn, k)] = d_.reindex(EVL), l_.reindex(EVL)
diff = pd.DataFrame(diff_rows).T[EVL]      # rows = (contrast, outcome), columns = dates
lev  = pd.DataFrame(lev_rows).T[EVL]
diff.index.names = lev.index.names = ["contrast", "outcome"]

draws_all = q(f"SELECT pool, draw_id, slot, fake_date FROM `{DS}.placebo_draws_strict`")
draws_all["fake_date"] = pd.to_datetime(draws_all.fake_date)
pl = ev[ev.event_set == "placebo"].set_index("event_date").event
POOL_EVENTS, DRAWS = {}, {}
for pool in POOLS:
    d = draws_all[draws_all.pool == pool]
    dates = sorted(d.fake_date.unique())
    missing = [x for x in dates if x not in pl.index]
    assert not missing, f"dates of pool {pool} not in cells: {missing}"
    POOL_EVENTS[pool] = [pl[x] for x in dates]
    pos = {x: j for j, x in enumerate(dates)}
    D = np.zeros((d.draw_id.nunique(), len(dates)))
    for r in d.itertuples():
        D[r.draw_id - 1, pos[r.fake_date]] += 1
    DRAWS[pool] = D / D.sum(1, keepdims=True)
    print(f"pool {pool}: {len(dates)} dates, {len(D)} draws")

# smoother, effects, ri_p, ri_ci as in 20
def smoother(pool, h, mask=None):
    idx = np.array([EV_IDX[e] for e in POOL_EVENTS[pool]])
    tp = tday[idx]
    ok = np.ones(len(idx), bool) if mask is None else mask
    lo = tp[ok].min() - 60
    S = np.zeros((len(ev), len(idx)))
    for i in range(len(ev)):
        w = np.exp(-0.5 * ((tp - tday[i]) / h) ** 2) * ok
        self_ = idx == i
        w[self_] = 0
        if tday[i] < lo:                           # before the pool: pool mean
            w = ok.astype(float); w[self_] = 0
            S[i] = w / w.sum(); continue
        X = np.column_stack([np.ones_like(tp), tp - tday[i]])
        A = X.T @ (X * w[:, None])
        S[i] = np.linalg.solve(A, (X * w[:, None]).T)[0]
    return S

def effects(Th, pool, h):
    idx = [EV_IDX[e] for e in POOL_EVENTS[pool]]
    Y = Th.to_numpy(float)
    P = Y[:, idx]
    B = np.full_like(Y, np.nan)
    nanmask = np.isnan(P)
    keys = pd.Series([m.tobytes() for m in nanmask])
    for key, rows_ in keys.groupby(keys).groups.items():
        rows_ = np.array(list(rows_))
        ok = ~nanmask[rows_[0]]
        if ok.sum() < 5:
            continue
        S = smoother(pool, h, ok)
        B[rows_] = Y[rows_] - np.nan_to_num(P[rows_]) @ S.T
    R = B[:, idx]
    return pd.DataFrame(B, index=Th.index, columns=Th.columns), R

def ri_p(beta, R):
    b = np.abs(np.asarray(beta, float))[:, :, None]
    r = np.abs(R)[:, None, :]
    cnt = np.sum(r >= b - 1e-15, axis=2)
    n = np.sum(~np.isnan(R), axis=1)[:, None]
    p = (1 + cnt) / (1 + n)
    p[np.isnan(np.asarray(beta, float))] = np.nan
    return p

def ri_ci(beta, R, a=0.10):
    lo_q, hi_q = np.nanquantile(R, 1 - a / 2, axis=1), np.nanquantile(R, a / 2, axis=1)
    return beta - lo_q[:, None], beta - hi_q[:, None]

# equal-tailed p-value: p_eq < 0.10 iff the 90% interval excludes 0
def p_eq(beta, R):
    b = np.asarray(beta, float)[:, :, None]
    r = R[:, None, :]
    ok = ~np.isnan(r)
    n = ok.sum(axis=2)
    ge = np.sum((r >= b - 1e-15) & ok, axis=2)
    le = np.sum((r <= b + 1e-15) & ok, axis=2)
    p = np.minimum(1.0, 2 * np.minimum(ge, le) / n)
    p[np.isnan(np.asarray(beta, float))] = np.nan
    return p

MAIN = ev[ev.event_set == "main"].event.tolist()
REPORT = ev[ev.event_set.isin(["main", "secondary", "nonquantum_news", "positive_control"])].event.tolist()
eff_rows, pooled_rows = [], []
for pool in POOLS:
    for h in BANDWIDTHS:
        B, R = effects(diff, pool, h)
        Bm = B[REPORT].to_numpy()
        p, pe = ri_p(Bm, R), p_eq(Bm, R)
        lo, hi = ri_ci(Bm, R)
        for j_, e in enumerate(REPORT):
            eff_rows.append(pd.DataFrame({"contrast": diff.index.get_level_values(0), "outcome": diff.index.get_level_values(1),
                                          "event": e, "pool": pool, "h": h, "theta": diff[e].values, "beta": B[e].values,
                                          "p_ri": p[:, j_], "p_eq": pe[:, j_], "ci90_lo": lo[:, j_], "ci90_hi": hi[:, j_],
                                          "level_T": lev[e].values}))
        mb = B[MAIN].mean(axis=1).to_numpy()
        null = np.nan_to_num(R) @ DRAWS[pool].T
        pp = (1 + np.sum(np.abs(null) >= np.abs(mb)[:, None] - 1e-15, axis=1)) / (1 + null.shape[1])
        ppe = np.minimum(1.0, 2 * np.minimum(np.mean(null >= mb[:, None] - 1e-15, axis=1),
                                             np.mean(null <= mb[:, None] + 1e-15, axis=1)))
        q05, q95 = np.quantile(null, 0.05, axis=1), np.quantile(null, 0.95, axis=1)
        pooled_rows.append(pd.DataFrame({"contrast": diff.index.get_level_values(0), "outcome": diff.index.get_level_values(1),
                                         "pool": pool, "h": h, "beta_mean": mb, "p_ri": pp, "p_eq": ppe,
                                         "ci90_lo": mb - q95, "ci90_hi": mb - q05,
                                         "null_mean": null.mean(axis=1), "level_T_mean": lev[MAIN].mean(axis=1).to_numpy()}))
eff = pd.concat(eff_rows, ignore_index=True).merge(ev[["event", "event_set", "event_date"]], on="event", how="left")
pooled = pd.concat(pooled_rows, ignore_index=True)
save(eff, "q22b_effects.csv", index=False)
save(pooled, "q22b_pooled.csv", index=False)

# regression checks against 20 and 21
def pick_eff(cn, o, e, col="beta", pool="main", h=H_MAIN):
    return eff[(eff.contrast == cn) & (eff.outcome == o) & (eff.event == e) & (eff.pool == pool) & (eff.h == h)][col].iloc[0]
def pick_pool(cn, o, col="beta_mean", pool="main", h=H_MAIN):
    return pooled[(pooled.contrast == cn) & (pooled.outcome == o) & (pooled.pool == pool) & (pooled.h == h)][col].iloc[0]
checks = {
    "20 pooled effect (L−C)":          (pick_pool("LC", "v_all"), REF_POOLED),
    "21 Randstorm θ (RS2)":            (pick_eff("UC_RS2", "v_all", "randstorm", "theta"), REF_RS2_THETA),
    "21 Randstorm β (RS2)":            (pick_eff("UC_RS2", "v_all", "randstorm"), REF_RS2_BETA),
    "21 Randstorm θ (RS2_all)":        (pick_eff("UC_RS2_all", "v_all", "randstorm", "theta"), REF_RS2ALL_THETA),
    "21 main events, pooled U−C":      (pick_pool("UC_RS2", "v_all"), REF_UC_POOLED),
}
for k, (got, ref) in checks.items():
    print(f"{k}: {got:.10f}   reference {ref:.10f}")
    assert abs(got - ref) < 1e-8, f"{k} does not match"
print("Reproduction passed: v_all matches 20 and 21 exactly")

# main table: Randstorm (U-C), pooled quantum events (L-C, U-C), six outcomes, pp
rows = []
for o in OUTC:
    r_ = eff[(eff.contrast == "UC_RS2") & (eff.outcome == o) & (eff.event == "randstorm") & (eff.pool == "main") & (eff.h == H_MAIN)].iloc[0]
    q_ = pooled[(pooled.contrast == "LC") & (pooled.outcome == o) & (pooled.pool == "main") & (pooled.h == H_MAIN)].iloc[0]
    u_ = pooled[(pooled.contrast == "UC_RS2") & (pooled.outcome == o) & (pooled.pool == "main") & (pooled.h == H_MAIN)].iloc[0]
    rows.append({"outcome": o,
                 "rs_beta": r_.beta * 100, "rs_ci90_lo": r_.ci90_lo * 100, "rs_ci90_hi": r_.ci90_hi * 100,
                 "rs_p_ri": r_.p_ri, "rs_p_eq": r_.p_eq, "rs_treated_level": r_.level_T * 100,
                 "quantum_LC_beta": q_.beta_mean * 100, "quantum_LC_ci90_hi": q_.ci90_hi * 100, "quantum_LC_p_ri": q_.p_ri, "quantum_LC_p_eq": q_.p_eq,
                 "quantum_UC_beta": u_.beta_mean * 100, "quantum_UC_ci90_hi": u_.ci90_hi * 100,
                 "rs_beta_over_quantum_LC_upper": r_.beta / q_.ci90_hi if q_.ci90_hi > 0 else np.nan})
summary = pd.DataFrame(rows).set_index("outcome")
save(summary.reset_index(), "q22b_summary.csv", index=False)
display(summary.round(3))
print("rs = Randstorm (contrast of 21: uncompressed U vs compressed C, RS2); quantum = 6 main events pooled. beta, intervals and levels in percentage points.")
print("Last column: the Randstorm effect as a multiple of the upper end of the 90% interval of the quantum-warning effect.")

## Step 4 · Composition of y1-strict outflows

RS2 keys only; Randstorm window (days 0…+30) and pooled placebo dates, U and C separately:

1. y1-strict outflows by transaction pattern × fate, with payment-or-change split by whether any output goes to an exposed address or an exchange;
2. roles of the other outputs in the same transactions, and the distribution of output counts.

In [ ]:
need("q22_outrole")
# 4a: y1 strict by pattern x fate, U and C (BTC allocated by key share)
SQL_COMP_UC = f"""
WITH fd AS (
  SELECT event, rel_day, date FROM `{DS}.flow_days` WHERE rel_day BETWEEN 0 AND {POST_MAX}
),
pk AS ({PK_UC}
),
fo AS ({FATE_SQL}
),
kt AS (
  SELECT fd.event, f.key_id, f.txid, f.share
  FROM `{DS}.flow_key_tx` f
  JOIN fd ON fd.date = f.date
  WHERE f.y1_strict > 0 AND f.date BETWEEN DATE '{LO}' AND DATE '{HI}'
    AND f.key_id IN (SELECT key_id FROM `{DS}.key_pubform` WHERE pubform IN ('U', 'C'))
)
SELECT pk.event, pk.event_set, pk.arm, pk.rec_old, fo.pattern, fo.fate,
  (fo.n_y0 = 0 AND fo.n_y2 = 0) AS noexp,
  COUNT(*) AS n_key_out, SUM(kt.share * fo.sats) / 1e8 AS btc
FROM kt
JOIN pk ON pk.event = kt.event AND pk.key_id = kt.key_id
JOIN fo ON fo.txid = kt.txid
GROUP BY 1, 2, 3, 4, 5, 6, 7
"""
comp_uc = q_guarded(SQL_COMP_UC, "q22b_comp_uc")
save(comp_uc, "q22b_composition_uc.csv", index=False)

# 4b: roles of the other outputs (allocated by key share)
SQL_TXMIX_UC = f"""
WITH fd AS (
  SELECT event, rel_day, date FROM `{DS}.flow_days` WHERE rel_day BETWEEN 0 AND {POST_MAX}
),
pk AS ({PK_UC}
),
tr AS ({TXROLE_SQL}
),
pat AS (SELECT txid, ANY_VALUE(pattern) AS pattern FROM `{DS}.q22_out` GROUP BY txid),
kt AS (
  SELECT fd.event, f.key_id, f.txid, f.share
  FROM `{DS}.flow_key_tx` f
  JOIN fd ON fd.date = f.date
  WHERE f.y1_strict > 0 AND f.date BETWEEN DATE '{LO}' AND DATE '{HI}'
    AND f.key_id IN (SELECT key_id FROM `{DS}.key_pubform` WHERE pubform IN ('U', 'C'))
)
SELECT pk.event, pk.event_set, pk.arm, pk.rec_old, pat.pattern,
  COUNT(*) AS n_key_tx,
  SUM(kt.share * tr.s_y1s) / 1e8 AS btc_y1s, SUM(kt.share * tr.s_change) / 1e8 AS btc_change,
  SUM(kt.share * tr.s_y0) / 1e8 AS btc_y0, SUM(kt.share * tr.s_y2) / 1e8 AS btc_y2,
  SUM(kt.share * tr.s_y1o) / 1e8 AS btc_y1o, SUM(kt.share * tr.s_oth) / 1e8 AS btc_oth,
  COUNTIF(tr.n_y0 > 0) AS n_with_y0, COUNTIF(tr.n_y2 > 0) AS n_with_y2, COUNTIF(tr.n_change > 0) AS n_with_change,
  COUNTIF(tr.n_out = 1) AS n_out1, COUNTIF(tr.n_out = 2) AS n_out2, COUNTIF(tr.n_out BETWEEN 3 AND 10) AS n_out3_10,
  COUNTIF(tr.n_out > 10) AS n_out_gt10
FROM kt
JOIN pk ON pk.event = kt.event AND pk.key_id = kt.key_id
JOIN tr ON tr.txid = kt.txid
JOIN pat ON pat.txid = kt.txid
GROUP BY 1, 2, 3, 4, 5
"""
txmix_uc = q_guarded(SQL_TXMIX_UC, "q22b_txmix_uc")
save(txmix_uc, "q22b_txmix_uc.csv", index=False)

# summary: Randstorm (RS2) and placebo dates (pooled), U and C
def tag(df):
    return np.where(df.event == "randstorm", "Randstorm", np.where(df.event_set == "placebo", "Placebo", "other"))
cu = comp_uc[~comp_uc.rec_old].assign(when=lambda d: tag(d))
cu = cu[cu.when != "other"]
cu["cls"] = np.where(cu.pattern == "coinjoin", "coinjoin",
             np.where(cu.pattern == "sweep", "sweep · " + cu.fate.replace({"to_exchange": "deposit", "consolidated": "deposit",
                                                                           "spent_unknown": "kept", "spent_kept": "kept"}),
                      "pay/change · " + np.where(cu.noexp, "no exposed/exchange output", "has exposed/exchange output")))
t_btc = cu.pivot_table(index=["when", "arm"], columns="cls", values="btc", aggfunc="sum", fill_value=0)
t_n = cu.pivot_table(index=["when", "arm"], columns="cls", values="n_key_out", aggfunc="sum", fill_value=0)
print("Composition of y1 strict (by BTC, %):");   display((t_btc.div(t_btc.sum(1), axis=0) * 100).round(1))
print("Composition of y1 strict (by key x output count, %):"); display((t_n.div(t_n.sum(1), axis=0) * 100).round(1))
print("Total BTC:"); display(t_btc.sum(1).round(1))
tm = txmix_uc[~txmix_uc.rec_old].assign(when=lambda d: tag(d))
tm = tm[tm.when != "other"].groupby(["when", "arm", "pattern"]).sum(numeric_only=True)
btc_cols = ["btc_y1s", "btc_change", "btc_y0", "btc_y2", "btc_y1o", "btc_oth"]
print("Value shares of output types in these transactions (%, allocated by key share):"); display((tm[btc_cols].div(tm[btc_cols].sum(1), axis=0) * 100).round(1))
print("Transaction characteristics (% of transactions):")
n_cols = ["n_with_y0", "n_with_y2", "n_with_change", "n_out1", "n_out2", "n_out3_10", "n_out_gt10"]
display((tm[n_cols].div(tm.n_key_tx, axis=0) * 100).round(1).join(tm.n_key_tx))

## Step 5 · Audit sample

Transactions with y1-strict outflows from U keys (RS2) in the Randstorm window, ordered by `FARM_FINGERPRINT(txid + seed)`: 30 payment-or-change, 10 sweep and up to 3 CoinJoin transactions. Each lists all outputs with script type, value, role, new-address flag, spend date, fate of y1-strict outputs and address.

Files: `rs_audit_sample.csv` (empty `manual_label` and `note` columns) and `rs_candidates.csv.gz` (all candidates).

In [ ]:
need("q22_outrole")
SQL_RS_CAND = f"""
WITH fd AS (
  SELECT date, rel_day FROM `{DS}.flow_days` WHERE event = 'randstorm' AND rel_day BETWEEN 0 AND {POST_MAX}
),
pk AS ({PK_UC}
),
cand AS (
  SELECT f.txid, f.key_id, f.date, fd.rel_day, f.share, f.key_in_sats, f.y1_strict
  FROM `{DS}.flow_key_tx` f
  JOIN fd ON fd.date = f.date
  JOIN pk ON pk.key_id = f.key_id AND pk.event = 'randstorm' AND pk.arm = 'U' AND NOT pk.rec_old
  WHERE f.y1_strict > 0 AND f.date BETWEEN DATE '{LO}' AND DATE '{HI}'
  QUALIFY ROW_NUMBER() OVER (PARTITION BY f.txid ORDER BY f.key_in_sats DESC) = 1
),
pat AS (SELECT txid, ANY_VALUE(pattern) AS pattern FROM `{DS}.q22_out` GROUP BY txid)
SELECT LOWER(TO_HEX(c.txid)) AS txid, LOWER(TO_HEX(c.key_id)) AS key_id, c.date, c.rel_day,
  ROUND(c.share, 4) AS share, ROUND(c.key_in_sats / 1e8, 8) AS key_in_btc, ROUND(c.y1_strict / 1e8, 8) AS y1_strict_btc,
  pat.pattern,
  ROW_NUMBER() OVER (PARTITION BY pat.pattern ORDER BY FARM_FINGERPRINT(CONCAT(TO_HEX(c.txid), '{SEED}'))) AS rk
FROM cand c
JOIN pat ON pat.txid = c.txid
"""
rs_cand = q_guarded(SQL_RS_CAND, "q22b_rs_cand")
print("Transactions with y1-strict outflows of group U (RS2) in the Randstorm window, by pattern:"); display(rs_cand.pattern.value_counts())
QUOTA = {"pay_or_change": N_PAY, "sweep": N_SWEEP, "coinjoin": N_CJ}
rs_s = rs_cand[rs_cand.rk <= rs_cand.pattern.map(QUOTA).fillna(0)].sort_values(["pattern", "rk"]).reset_index(drop=True)
rs_s.insert(0, "id", [f"R{i+1:02d}" for i in range(len(rs_s))])

txs = ", ".join(f"FROM_HEX('{t}')" for t in rs_s.txid)
dates = ", ".join(sorted({f"DATE '{pd.Timestamp(d).strftime('%Y-%m-%d')}'" for d in rs_s.date}))
outs = q_guarded(f"""
WITH fo AS ({FATE_SQL}
)
SELECT LOWER(TO_HEX(o.txid)) AS txid, o.vout, o.type, o.address, o.sats / 1e8 AS btc, o.role, o.is_new_addr,
  o.out_spent_date, o.n_in, o.n_in_keys, fo.fate
FROM `{DS}.q22_outrole` o
LEFT JOIN fo ON fo.txid = o.txid AND fo.vout = o.vout
WHERE o.created_date IN ({dates}) AND o.txid IN ({txs})
ORDER BY txid, vout
""", "q22b_rs_outputs")
ROLE_LABEL = {"change": "change", "y2": "exchange", "y0": "exposed", "y1s": "y1_strict", "y1o": "unexposed_other", "oth": "no_key"}
outs["desc"] = [f"{r.vout}:{r.type}:{r.btc:.8f}:{ROLE_LABEL[r.role]}:new_addr={str(bool(r.is_new_addr)).lower()}"
                f":spent={'' if pd.isna(r.out_spent_date) else pd.Timestamp(r.out_spent_date).strftime('%Y-%m-%d')}"
                + (f":fate={r.fate}" if isinstance(r.fate, str) else "")
                for r in outs.itertuples()]
g = outs.groupby("txid").agg(n_in=("n_in", "first"), n_in_keys=("n_in_keys", "first"), n_out=("vout", "size"),
                             out_btc=("btc", "sum"), outputs=("desc", " | ".join),
                             out_addresses=("address", lambda s: " | ".join(map(str, s))))
rs_s = rs_s.merge(g, left_on="txid", right_index=True, how="left")
assert rs_s.n_out.notna().all(), "outputs not found for some sampled transactions"
rs_s["url"] = "https://mempool.space/tx/" + rs_s.txid
for c in ["manual_label", "note"]:
    rs_s[c] = ""
cols = ["id", "txid", "url", "date", "rel_day", "pattern", "key_in_btc", "share", "y1_strict_btc",
        "n_in", "n_in_keys", "n_out", "out_btc", "outputs", "out_addresses", "manual_label", "note"]
save(rs_s[cols], "rs_audit_sample.csv", index=False)
save(rs_cand.drop(columns="rk"), "rs_candidates.csv.gz", index=False, compression="gzip")
print(f"Sampled {len(rs_s)} transactions:", rs_s.pattern.value_counts().to_dict())
display(rs_s[["id", "date", "pattern", "key_in_btc", "y1_strict_btc", "n_in", "n_out", "outputs"]].head(10))

In [ ]:
for rec in JOBS[-4:]:
    job = client.get_job(rec["job_id"], location=rec.get("location", LOCATION))
    print(rec["table"], job.state, "errors:", job.errors)
    if job.state != "DONE":
        job.result(); print("  done")
n = table_rows("q22_outrole")
print("q22_outrole", f"{n:,} rows" if n else "not built yet")

In [ ]:
report = {"generated_at": dt.datetime.now(dt.timezone.utc).isoformat(),
          "config": {"freeze": F, "flow_days": [LO, HI], "post_max": POST_MAX, "rs_win": RS_WIN, "outcomes": OUTC,
                     "h_main": H_MAIN, "bandwidths": BANDWIDTHS, "pools": POOLS, "seed": SEED,
                     "quota": {"pay_or_change": N_PAY, "sweep": N_SWEEP, "coinjoin": N_CJ}},
          "ledger": LEDGER,
          "tables": [table_info(t) for t in ["q22_outrole"] if table_rows(t)]}
for k, v in [("check_step1", "chk1"), ("checks_replication", "pd.DataFrame(checks, index=['got', 'ref']).T.reset_index()"),
             ("summary_pp", "summary.reset_index()"), ("rs_candidates_by_pattern", "rs_cand.pattern.value_counts().reset_index()"),
             ("randstorm_sizes", "sz.reset_index()")]:
    try: report[k] = eval(v).astype(str).to_dict(orient="records")
    except NameError: pass
with open(os.path.join(OUT_DIR, "build_report_22b.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2, default=str)
with zipfile.ZipFile("out22b.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, fn), arcname=f"out22b/{fn}")
print("Packed:", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files
    files.download("out22b.zip")
except Exception:
    print("out22b.zip is in the working directory")